# FIG-Loneliness Pipeline Runner (Colab → HF Spaces)

Run this notebook on **Google Colab (T4 GPU)** to:
1. Install dependencies
2. Clone the FIG-Loneliness repo
3. Run the full NLP pipeline (preprocessing → EDA → features → training → evaluation)
4. Upload results to your HuggingFace Space

## Before You Start

Set these secrets in Colab: **Edit → Secrets** (🔑 icon in left sidebar)
- `HF_TOKEN` — Your HuggingFace write token
- `HF_REPO_ID` — Your Space repo ID (e.g., `username/fig-lone`)

Then set the runtime to **T4 GPU**: Runtime → Change runtime type → T4 GPU

In [22]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — continuing on CPU (slower)')

Sun Mar 29 02:49:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
# Install Python dependencies
import subprocess
import sys

deps = [
    'spacy', 'ftfy', 'bleach', 'emoji', 'datasets', 'transformers',
    'scikit-learn', 'gensim', 'sentence-transformers',
    'matplotlib', 'wordcloud', 'umap-learn', 'seaborn',
    'fastapi', 'uvicorn', 'joblib', 'huggingface-hub'
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + deps)

# Install spaCy model
subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

Git LFS initialized.
PyTorch 2.10.0+cu128 | CUDA: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [17]:
import os
from google.colab import drive

# Mount Google Drive to persist results if needed
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/FIG'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Colab Notebooks/FIG/fig-lone


In [ ]:
# Clone GitHub repo
import subprocess
import os

REPO = 'https://github.com/codenameberyl/fig-lone.git'

if not os.path.exists('fig-lone'):
    subprocess.check_call(['git', 'clone', REPO, 'fig-lone'])
    os.chdir('fig-lone')
else:
    os.chdir('fig-lone')
    subprocess.check_call(['git', 'pull'])

print(f'Working in: {os.getcwd()}')

In [19]:
# Load HuggingFace credentials from Colab secrets
from google.colab import userdata

def get_secret(key, fallback=''):
    try:
        return userdata.get(key)
    except:
        return fallback

HF_TOKEN = get_secret('HF_TOKEN')
HF_REPO_ID = get_secret('HF_REPO_ID')  # e.g. username/fig-lone

if not HF_TOKEN or not HF_REPO_ID:
    print('⚠️  HF_TOKEN or HF_REPO_ID not set!')
    print('Set these in Colab Secrets: Edit → Secrets (🔑 icon)')
else:
    print(f'✓ HF repo: {HF_REPO_ID}')

HF repo: codenameberyl/fig-lone-api
S3 bucket: not configured


In [20]:
import sys
sys.path.insert(0, '.')
print('✓ Package path configured')

Package path set


In [23]:

# Run the full pipeline (this will take 45-90 min with T4 GPU)
import subprocess
print('Starting FIG-Loneliness pipeline...')
print('This may take 45-90 minutes with T4 GPU\n')
subprocess.check_call(['python', '-m', 'run_pipeline'])
print('\n✓ Pipeline complete!')

02:50:20 | INFO     | __main__ — ============================================================
02:50:20 | INFO     | __main__ —   FIG-Loneliness NLP Pipeline
02:50:20 | INFO     | __main__ — ============================================================
02:50:20 | INFO     | __main__ — ────────────────────────────────────────────────────────────
02:50:20 | INFO     | __main__ —   STAGE: LOAD
02:50:20 | INFO     | __main__ — ────────────────────────────────────────────────────────────
02:50:20 | INFO     | src.dataset_loader — Downloading dataset from FIG-Loneliness/FIG-Loneliness...
02:50:21 | INFO     | httpx — HTTP Request: GET https://huggingface.co/api/datasets/FIG-Loneliness/FIG-Loneliness/revision/main "HTTP/1.1 200 OK"
Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]02:50:21 | INFO     | httpx — HTTP Request: HEAD https://huggingface.co/datasets/FIG-Loneliness/FIG-Loneliness/resolve/1d89c0235a300f314ddb4fd33d779d57eb24b63c/dev_set/dataset_info.json "HTTP/1.1 307 Temporary Redirect"
02:5

In [ ]:
# Inspect artifact sizes before upload
import fnmatch
from pathlib import Path

results_dir = Path('results')

# Cache allowlist: only upload essential models and feature vectors
CACHE_ALLOWLIST = [
    'tfidf_vectorizer.joblib',
    'sbert_model.joblib',
    'word2vec_model.joblib',
    '*_logistic_regression.joblib',
    '*_svm.joblib',
    '*_random_forest.joblib',
    'distilbert_results.joblib',
    'all_model_results.joblib',
    'best_per_representation.joblib',
]

def should_upload(path: Path, results_dir: Path) -> bool:
    """Determine if a file should be uploaded to HF Spaces."""
    rel = str(path.relative_to(results_dir)).replace('\\', '/')
    
    # All JSON results
    if rel.startswith('json/'):
        return True
    # All plots
    if rel.startswith('plots/'):
        return True
    # DistilBERT: only best_model/
    if rel.startswith('bert/'):
        return rel.startswith('bert/best_model/')
    # Cache: only allowlisted files
    if rel.startswith('cache/'):
        return any(fnmatch.fnmatch(path.name, pat) for pat in CACHE_ALLOWLIST)
    return False

# Analyze files
all_files = [f for f in results_dir.rglob('*') if f.is_file()]
upload_files = [f for f in all_files if should_upload(f, results_dir)]
skip_files = [f for f in all_files if not should_upload(f, results_dir)]

upload_mb = sum(f.stat().st_size for f in upload_files) / 1e6
skip_mb = sum(f.stat().st_size for f in skip_files) / 1e6

print(f'WILL UPLOAD:  {len(upload_files):4d} files  {upload_mb:8.1f} MB')
for subdir in sorted(set(f.relative_to(results_dir).parts[0] for f in upload_files)):
    files_in_subdir = [f for f in upload_files if subdir in f.relative_to(results_dir).parts]
    size_mb = sum(f.stat().st_size for f in files_in_subdir) / 1e6
    print(f'  {subdir:20s}  {len(files_in_subdir):3d} files  {size_mb:8.1f} MB')

print(f'\nWILL SKIP:    {len(skip_files):4d} files  {skip_mb:8.1f} MB')
print('  (checkpoints, dataset, feature matrices — not needed for API)')

WILL UPLOAD    50 files     425.4 MB
  bert/best_model           4 files     268.5 MB
  cache                    17 files     154.7 MB
    all_model_results.joblib
    distilbert_results.joblib
    sbert_logistic_regression.joblib
    sbert_model.joblib
    sbert_random_forest.joblib
    sbert_svm.joblib
    tfidf_ling_logistic_regression.joblib
    tfidf_ling_random_forest.joblib
    tfidf_ling_svm.joblib
    tfidf_logistic_regression.joblib
    tfidf_random_forest.joblib
    tfidf_svm.joblib
    tfidf_vectorizer.joblib
    word2vec_logistic_regression.joblib
    word2vec_model.joblib
    word2vec_random_forest.joblib
    word2vec_svm.joblib
  json                     16 files       0.2 MB
  plots                    13 files       2.0 MB

WILL SKIP      36 files    2580.4 MB
  bert/checkpoint-247       8 files     803.6 MB
  bert/checkpoint-494       8 files     803.6 MB
  bert/checkpoint-741       8 files     803.6 MB
  cache                    12 files     169.7 MB


In [ ]:
# Upload results to HuggingFace Spaces
from huggingface_hub import HfApi
from pathlib import Path
import fnmatch

if not HF_TOKEN or not HF_REPO_ID:
    print('❌ HF_TOKEN or HF_REPO_ID not set. Skipping upload.')
else:
    print(f'Uploading to HF Space: {HF_REPO_ID}\n')
    
    api = HfApi(token=HF_TOKEN)
    results_dir = Path('results')
    
    # Collect files to upload
    files_to_upload = sorted([
        f for f in results_dir.rglob('*')
        if f.is_file() and should_upload(f, results_dir)
    ])
    
    total_mb = sum(f.stat().st_size for f in files_to_upload) / 1e6
    print(f'Uploading {len(files_to_upload)} files ({total_mb:.1f} MB)...\n')
    
    for i, local_path in enumerate(files_to_upload, 1):
        rel_path = str(local_path.relative_to(results_dir)).replace('\\', '/')
        remote_path = f'data/resultsl_path}'
        
        try:
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=remote_path,
                repo_id=HF_REPO_ID,
                repo_type='space',
                token=HF_TOKEN,
            )
            if i % 5 == 0 or i == len(files_to_upload):
                print(f'  [{i}/{len(files_to_upload)}] {rel_path}')
        except Exception as e:
            print(f'  ⚠️  Failed to upload {rel_path}: {e}')
    
    print(f'\n✓ Upload complete! ({len(files_to_upload)} files, {total_mb:.1f} MB)')

Files to upload: 50 (425.4 MB)
Breakdown:
  bert/best_model         4 files     268.5 MB
  cache                  17 files     154.7 MB
  json                   16 files       0.2 MB
  plots                  13 files       2.0 MB

  5/50 — cache/all_model_results.joblib
  10/50 — cache/sbert_svm.joblib
  15/50 — cache/tfidf_random_forest.joblib
  20/50 — cache/word2vec_random_forest.joblib
  25/50 — json/eda_class_distribution.json
  30/50 — json/eda_summary.json
  35/50 — json/pipeline_state.json
  40/50 — plots/eda_correlation_heatmap.png
  45/50 — plots/eda_wordcloud_lonely.png
  50/50 — plots/eval_roc_distilbert_distilbert.png

✓ Upload complete — 50 files (425.4 MB)
Next step: run the verify cell to reboot and confirm the API is live.


In [ ]:
# Verify API is running after artifact upload
import requests
import time

if not HF_TOKEN or not HF_REPO_ID:
    print('Skipping verification (credentials not set)')
else:
    username, space_name = HF_REPO_ID.split('/')
    api_url = f'https://{username}-{space_name}.hf.space/api/status'
    
    print(f'Testing API at {api_url}')
    print('Waiting for Space to ingest results...\n')
    
    for attempt in range(5):
        try:
            r = requests.get(api_url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                print(f'✓ API is online!')
                print(f'  Status: {data.get("status")}')
                print(f'  Title: {data.get("title")}')
                print(f'\n📖 Interactive docs: https://{username}-{space_name}.hf.space/docs')
                print(f'🚀 Try prediction: POST https://{username}-{space_name}.hf.space/api/predict')
                break
            else:
                print(f'  Attempt {attempt + 1}/5: HTTP {r.status_code}')
        except Exception as e:
            print(f'  Attempt {attempt + 1}/5: {str(e)[:50]}...')
        
        if attempt < 4:
            time.sleep(15)

Triggering Space rebuild (this picks up the uploaded artifacts)...
  Rebuild triggered. HF is now rebuilding the Docker image.
  This takes 3–8 minutes. You can watch progress at:
  https://huggingface.co/spaces/codenameberyl/fig-lone-api

  Waiting 5 minutes for build + startup...
  Done waiting.                

Checking https://codenameberyl-fig-lone-api.hf.space/api/status ...
  status   : ok
  pipeline : 0 steps done — []

  API is up but pipeline_state not loaded yet.
  Check the Space build logs — it may still be building.
  https://huggingface.co/spaces/codenameberyl/fig-lone-api


## Done! ✓

Your pipeline results have been uploaded to your HuggingFace Space.

### Next Steps

1. **Check your API** at: `https://{username}-fig-lone.hf.space/docs`
2. **Try a prediction**:
   ```bash
   curl -X POST https://{username}-fig-lone.hf.space/api/predict \
     -H "Content-Type: application/json" \
     -d '{"text": "I feel so alone and isolated"}'
   ```
3. **View results**:
   - `/api/eda/*` — Exploratory analysis
   - `/api/models/*` — Model metrics and confusion matrices
   - `/api/status` — Pipeline state

### Re-run Later

To run this pipeline again and push updated results:
1. Open this notebook again
2. Update to latest code (if needed): `git pull`
3. Run all cells
4. Results are automatically overwritten — no redeploy needed